# NAVIER 001.5 — Auditoría técnica y estadística (borrador)

**Propósito:** auditar V0.1 sin modificar sus resultados ni afirmar que la simulación demuestra señal en mercados reales. Este notebook registra hallazgos y ejecuta diagnósticos reproducibles.

**Regla de investigación:** no cambiar umbrales después de ver resultados sin documentar una nueva versión. No conectar todavía a Kraken.

**Orden:** entorno → integridad de datos/features → variables inspiradas en Navier → validación temporal → registro de decisiones.


## 0. Entorno y trazabilidad

Ejecuta desde la raíz del proyecto después de instalar `requirements.txt`. Si falta una dependencia, registra el error en lugar de alterar silenciosamente el entorno.


In [1]:
import sys, platform, importlib.metadata as md
from pathlib import Path
import numpy as np
import pandas as pd

print("Python:", sys.version)
print("Platform:", platform.platform())
for pkg in ["numpy", "pandas", "pyarrow", "scipy", "statsmodels", "matplotlib", "nbformat"]:
    try: print(f"{pkg}: {md.version(pkg)}")
    except md.PackageNotFoundError: print(f"{pkg}: MISSING")


Python: 3.10.9 (tags/v3.10.9:1dd9be6, Dec  6 2022, 20:01:21) [MSC v.1934 64 bit (AMD64)]
Platform: Windows-10-10.0.26300-SP0
numpy: 2.2.6
pandas: 2.3.3
pyarrow: 24.0.0
scipy: 1.15.3
statsmodels: 0.15.0
matplotlib: 3.10.9
nbformat: 5.11.1


## 1. Cargar el pipeline sintético

Esta auditoría usa datos sintéticos incluidos en el ZIP. No se descargan datos de mercado.


In [2]:
from navier_lab import synthetic as S, features as F, analysis as A, collector as C
from navier_lab.schema import DEFAULT_CONFIG as CFG, RAW_COLUMNS, BID_QTY, ASK_QTY, BID_PX, ASK_PX

raw, latent = S.generate("S2_navier", n=12000, seed=17)
feat = F.compute_features(raw, CFG, with_target=True)
print("raw shape:", raw.shape, "features shape:", feat.shape)
print("timestamp monotonic:", raw.timestamp.is_monotonic_increasing)
print("feature columns:", len(F.FEATURE_COLUMNS))


raw shape: (12000, 41) features shape: (12000, 31)
timestamp monotonic: True
feature columns: 24


## 2. Integridad de datos y valores no finitos

La validación actual revisa algunas condiciones, pero esta celda amplía el diagnóstico: precios/cantidades finitos, orden de niveles y valores finitos de features. No repara datos; informa problemas.


In [3]:
def audit_raw(df):
    report = {}
    report["missing_columns"] = sorted(set(RAW_COLUMNS) - set(df.columns))
    numeric = [c for c in RAW_COLUMNS if c != "timestamp"]
    report["nonfinite_numeric_cells"] = int((~np.isfinite(df[numeric].to_numpy(float))).sum())
    report["nonpositive_qty_cells"] = int((df[BID_QTY + ASK_QTY] <= 0).sum().sum())
    report["crossed_or_locked_top"] = int((df[ASK_PX[0]] <= df[BID_PX[0]]).sum())
    report["bid_levels_not_descending"] = int((df[BID_PX].to_numpy()[:, :-1] < df[BID_PX].to_numpy()[:, 1:]).any(axis=1).sum())
    report["ask_levels_not_ascending"] = int((df[ASK_PX].to_numpy()[:, :-1] > df[ASK_PX].to_numpy()[:, 1:]).any(axis=1).sum())
    report["timestamp_duplicates"] = int(df.timestamp.duplicated().sum())
    report["timestamp_gaps_or_irregular"] = int((df.timestamp.diff().dropna() != pd.Timedelta("1s")).sum())
    return pd.Series(report, dtype=object)

audit_raw(raw)


missing_columns                []
nonfinite_numeric_cells         0
nonpositive_qty_cells           0
crossed_or_locked_top           0
bid_levels_not_descending       0
ask_levels_not_ascending        0
timestamp_duplicates            0
timestamp_gaps_or_irregular     0
dtype: object

In [4]:
numeric_features = [c for c in F.FEATURE_COLUMNS if c in feat.columns]
nonfinite = pd.DataFrame({
    "nan": feat[numeric_features].isna().sum(),
    "posinf": np.isposinf(feat[numeric_features].to_numpy(float)).sum(axis=0),
    "neginf": np.isneginf(feat[numeric_features].to_numpy(float)).sum(axis=0),
}).sort_values(["posinf", "neginf", "nan"], ascending=False)
nonfinite.head(20)


,nan,posinf,neginf
vol60_bps,60,0,0
re_log,60,0,0
ofi_5,59,0,0
ret_30,30,0,0
imb_full_lag8,8,0,0
dI_dt,7,0,0
ret_5,5,0,0
imb_full_lag5,5,0,0
u,4,0,0
imb_full_lag2,2,0,0


## 3. Auditoría de gradientes y curvaturas

En `features.py`, la normalización actual usa la media de las 20 cantidades (10 niveles bid + 10 ask) por fila. Las bases polinómicas son centradas y ortogonales bajo el producto interno uniforme de niveles. Eso no implica que el resultado tenga una interpretación física validada.

La celda siguiente verifica que la escala no explote ante un nivel con tamaño muy grande y compara la normalización actual con una normalización robusta de diagnóstico. La normalización robusta es **sólo diagnóstico**, no reemplaza la feature original.


In [5]:
bq = raw[BID_QTY].to_numpy(float); aq = raw[ASK_QTY].to_numpy(float)
vol20_mean = np.concatenate([bq, aq], axis=1).mean(axis=1)
vol20_median = np.median(np.concatenate([bq, aq], axis=1), axis=1)
print("mean denominator quantiles:", np.quantile(vol20_mean, [0, .5, .95, .99, 1]))
print("median denominator quantiles:", np.quantile(vol20_median, [0, .5, .95, .99, 1]))
print("grad_bid quantiles:", feat.grad_bid.quantile([0,.01,.5,.99,1]).to_dict())
print("grad_ask quantiles:", feat.grad_ask.quantile([0,.01,.5,.99,1]).to_dict())


mean denominator quantiles: [0.59081822 0.89332533 1.19676447 1.51547773 2.51372147]
median denominator quantiles: [0.33205029 0.75219788 0.95914331 1.08051697 1.39339787]
grad_bid quantiles: {0.0: -0.4105506537360634, 0.01: -0.1963318914462186, 0.5: 0.05855160974269498, 0.99: 0.40184889863556134, 1.0: 0.6962300262829964}
grad_ask quantiles: {0.0: -0.40965651291701155, 0.01: -0.17851970152391564, 0.5: 0.07156929144478924, 0.99: 0.4137948290987429, 1.0: 0.69819829565973}


### Interpretación que debe quedar documentada

- El signo de `grad_bid` depende de cómo se indexan los niveles: aquí el índice aumenta del mejor precio hacia niveles más alejados. Un gradiente positivo indica, en promedio, cantidades normalizadas mayores en niveles más alejados; no “más liquidez cerca del BBO”.
- La normalización compartida por ambos lados introduce dependencia entre gradientes bid/ask y el tamaño total del libro.
- `curv_*` mide una proyección cuadrática del perfil de tamaños por nivel; no es curvatura espacial física.


## 4. Auditoría de `Re_m` / `re_log`

Implementación actual: `abs(u) * L_bps / vol60_bps**2`, donde `L_bps` es la distancia ask nivel 10 menos bid nivel 10 en bps y `vol60_bps` es la desviación estándar móvil de retornos de 1 segundo. Esta razón **no debe describirse como adimensional sin una convención temporal explícita**. También puede ser inestable cuando la volatilidad es cero o casi cero.


In [6]:
vol = feat["vol60_bps"].to_numpy(float)
re = feat["re_m"].to_numpy(float)
report = {
    "vol_zero_or_near_zero_count": int(np.sum(np.isfinite(vol) & (vol < 1e-8))),
    "re_nonfinite_count": int((~np.isfinite(re)).sum()),
    "re_negative_count": int(np.sum(re < 0)),
    "re_quantiles_finite": np.quantile(re[np.isfinite(re)], [0,.5,.95,.99,1]).tolist() if np.isfinite(re).any() else None,
    "corr_re_log_vs_vol": float(pd.Series(feat["re_log"]).corr(pd.Series(vol), method="spearman")),
}
report


{'vol_zero_or_near_zero_count': 0,
 're_nonfinite_count': 60,
 're_negative_count': 0,
 're_quantiles_finite': [2.5718652548030137e-08,
  0.2438560060274588,
  0.877793727010437,
  1.2876214131700756,
  2.156253453465963],
 'corr_re_log_vs_vol': -0.2623763505697844}

### Decisión pendiente — no cambiar fórmula todavía

Antes de elegir un piso para la volatilidad o una transformación robusta, registrar: (a) definición operacional de `L`; (b) unidades temporales; (c) frecuencia de valores no finitos; (d) sensibilidad a episodios de volatilidad baja; (e) si `re_log` aporta información incremental después de controlar `vol60_bps` y profundidad. Toda variante nueva debe tener un nombre/versionado separado.


## 5. `dI_dt`: correlación no equivale a redundancia

No se exige ortogonalidad por definición. La pregunta es si la feature mejora predicción fuera de muestra una vez controlado el baseline B+. Primero medimos correlaciones y condicionamiento, como diagnóstico descriptivo.


In [7]:
cols = [c for c in ["imb_full", "imb_full_lag2", "imb_full_lag5", "imb_full_lag8", "dI_dt"] if c in feat]
corr = feat[cols].corr(method="spearman")
corr


,imb_full,imb_full_lag2,imb_full_lag5,imb_full_lag8,dI_dt
imb_full,1.000000,0.953655,0.891366,0.835289,0.235563
imb_full_lag2,0.953655,1.000000,0.932190,0.871433,0.144869
imb_full_lag5,0.891366,0.932190,1.000000,0.932189,-0.148109
imb_full_lag8,0.835289,0.871433,0.932189,1.000000,-0.229478
dI_dt,0.235563,0.144869,-0.148109,-0.229478,1.000000


## 6. Target y chequeo causal de features

Se verifica que el target futuro coincida con la definición en el código y que modificar filas posteriores a `t0` no altere features calculadas hasta `t0`. Esta prueba no sustituye una auditoría de todos los posibles flujos de información.


In [8]:
H = CFG.horizon
manual_y_bps = 1e4 * (np.log(feat["mid"].shift(-H)) - np.log(feat["mid"]))
print("target max abs diff (bps):", float(np.nanmax(np.abs(manual_y_bps - feat[F.TARGET_BPS]))))

t0 = len(raw) // 2
base = F.compute_features(raw, CFG, with_target=False)
mut = raw.copy()
rng = np.random.default_rng(2026)
for col in [c for c in raw.columns if c != "timestamp"]:
    vals = mut.loc[t0+1:, col].to_numpy(float)
    mut.loc[t0+1:, col] = vals * (1 + 0.03 * rng.standard_normal(len(vals)))
changed = F.compute_features(mut, CFG, with_target=False)
compare_cols = F.FEATURE_COLUMNS + ["mid"]
maxdiff = np.nanmax(np.abs(base.loc[:t0, compare_cols].to_numpy(float) - changed.loc[:t0, compare_cols].to_numpy(float)))
print("max feature difference through t0 after mutating future:", float(maxdiff))
assert maxdiff < 1e-9


target max abs diff (bps): 1.4381384971784428e-11


max feature difference through t0 after mutating future: 0.0


C:\Users\USER\AppData\Local\Temp\ipykernel_1184\1487317710.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.15234075 0.13909022 0.13176313 ... 0.25309382 0.21987532 0.23308473]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  mut.loc[t0+1:, col] = vals * (1 + 0.03 * rng.standard_normal(len(vals)))
C:\Users\USER\AppData\Local\Temp\ipykernel_1184\1487317710.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[0.29918763 0.23584571 0.22841757 ... 0.37848702 0.50609332 0.58493382]' has dtype incompatible with float32, please explicitly cast to a compatible dtype first.
  mut.loc[t0+1:, col] = vals * (1 + 0.03 * rng.standard_normal(len(vals)))
C:\Users\USER\AppData\Local\Temp\ipykernel_1184\1487317710.py:11: FutureWarning: Setting an item of incompatible dtype is deprecated and

## 7. Auditoría del purged walk-forward

La implementación actual usa `purge=H` por defecto. Para un target de retorno futuro a horizonte `H`, el objetivo de una observación de entrenamiento en `t` termina en `t+H`. Debemos garantizar que ese intervalo no se solape con el test. Las ventanas retrospectivas de features pueden compartir observaciones históricas sin constituir fuga por sí mismas si se calculan causalmente.

Esta celda comprueba la separación efectiva entre la última etiqueta de train y el inicio de test.


In [9]:
splits = A.wf_splits(len(feat.dropna(subset=F.FEATURE_COLUMNS + [F.TARGET]).reset_index(drop=True)), K=6, purge=H)
rows=[]
for i,(tr,te) in enumerate(splits,1):
    rows.append({"fold":i,"train_last_idx":int(tr.max()),"test_first_idx":int(te.min()),
                 "index_gap":int(te.min()-tr.max()-1),
                 "label_end_last_train":int(tr.max()+H),
                 "label_overlap_test":bool(tr.max()+H >= te.min())})
pd.DataFrame(rows)


,fold,train_last_idx,test_first_idx,index_gap,label_end_last_train,label_overlap_test
0,1,1977,1988,10,1987,False
1,2,3965,3976,10,3975,False
2,3,5954,5965,10,5964,False
3,4,7942,7953,10,7952,False
4,5,9930,9941,10,9940,False


## 8. Hallazgos iniciales y registro de decisiones

Completar después de ejecutar todas las celdas. No convertir un diagnóstico en corrección automática.

| ID | Hallazgo | Severidad provisional | Acción antes de datos reales |
|---|---|---|---|
| A-01 | `Re_m` tiene una interpretación dimensional ambigua y puede ser inestable con volatilidad cercana a cero | Alta | Medir no finitos y documentar unidades; evaluar variantes versionadas |
| A-02 | `L_bps` es el ancho entre nivel 10 ask y nivel 10 bid, no una medida universal de profundidad | Media | Renombrar/documentar o comparar definiciones alternativas |
| A-03 | La validación de snapshots no comprueba finitud ni orden completo de niveles | Alta | Añadir pruebas específicas antes del recolector real |
| A-04 | `to_1s_grid` no conserva el segundo parcial entre chunks | Alta para streaming real | Rediseñar resampleo incremental |
| A-05 | La purga debe auditarse en función del intervalo de etiquetas y del diseño temporal, no imponer `H+W` automáticamente | Alta | Probar folds con índices absolutos y target end-time |
| A-06 | `dI_dt` puede correlacionarse con imbalance y rezagos sin ser automáticamente inválida | Media | Evaluar aporte incremental y estabilidad OOS |

**Gate para pasar a datos reales:** pruebas pasan; no hay features no finitas sin política explícita; folds no solapan etiquetas; protocolo de datos L2 y reconexión está definido; métricas y umbrales quedan pre-registrados.
